# Chapter 17 &mdash; Canonicity via Myhill&ndash;Nerode, Hash Consing, and Apply

**Concept 6 of the Chapter 17 decomposition:** *Canonicity via Myhill–Nerode, Hash Consing, and the Apply Operation*

BDDs for a function are isomorphic given a variable order, so equality checking is a pointer comparison.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17/Concept-Canonicity-And-Apply/Concept-Canonicity-And-Apply.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Bdd            import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The payoff of the whole construction.

**Canonicity.** For a fixed variable order, two reduced BDDs represent the same
function **iff they are isomorphic**. This is Myhill&ndash;Nerode (Chapter 6, Concept 7)
transplanted: the minimal DFA of a language is unique, so the reduced BDD of a
function is too.

**Hash consing** turns isomorphism into **pointer equality**. Because `mk` never
creates a duplicate, two structurally identical BDDs are the *same object*, and
equivalence checking is $O(1)$ rather than a graph traversal.

**Apply** combines two BDDs under a binary operation by recursing on the top variable
and memoising on the pair of nodes. With the cache it runs in $O(|f|\cdot|g|)$ &mdash;
which is why BDD packages can do real work.

Together: tautology checking, equivalence checking and satisfiability all become
trivial *given* the BDD.

## 2. Definitions

### The BDD package

In [ ]:
# --- a minimal BDD package ----------------------------------------------
# A node is either the terminal 0/1, or ('n', var_index, low, high) where
# low is the 0-branch and high the 1-branch.  Hash consing (the `unique`
# table) is what makes the representation canonical: structurally equal
# subgraphs become the SAME Python object, so equality is pointer equality.
ZERO, ONE = 0, 1

class BDD:
    def __init__(self, nvars):
        self.nvars = nvars
        self.unique = {}          # (var, low, high) -> node  -- hash consing
        self.apply_cache = {}

    def mk(self, var, low, high):
        if low is high: return low            # REDUCTION 1: skip a useless test
        key = (var, id(low), id(high), self._k(low), self._k(high))
        if key in self.unique: return self.unique[key]   # REDUCTION 2: share
        node = ('n', var, low, high)
        self.unique[key] = node
        return node

    def _k(self, n):
        return n if n in (ZERO, ONE) else ('n', n[1], self._k(n[2]), self._k(n[3]))

    def var(self, i):
        return self.mk(i, ZERO, ONE)

    def apply(self, op, a, b):
        key = (op, self._k(a), self._k(b))
        if key in self.apply_cache: return self.apply_cache[key]
        if a in (ZERO, ONE) and b in (ZERO, ONE):
            r = ONE if op(bool(a), bool(b)) else ZERO
        else:
            va = a[1] if a not in (ZERO, ONE) else self.nvars
            vb = b[1] if b not in (ZERO, ONE) else self.nvars
            v = min(va, vb)
            al, ah = (a[2], a[3]) if va == v else (a, a)
            bl, bh = (b[2], b[3]) if vb == v else (b, b)
            r = self.mk(v, self.apply(op, al, bl), self.apply(op, ah, bh))
        self.apply_cache[key] = r
        return r

    def NOT(self, a):  return self.apply(lambda x, y: not x, a, a)
    def AND(self, a, b): return self.apply(lambda x, y: x and y, a, b)
    def OR(self, a, b):  return self.apply(lambda x, y: x or y, a, b)
    def XOR(self, a, b): return self.apply(lambda x, y: x != y, a, b)

    def evaluate(self, node, assign):
        while node not in (ZERO, ONE):
            node = node[3] if assign[node[1]] else node[2]
        return bool(node)

    def size(self, node):
        seen = set()
        def walk(n):
            if n in (ZERO, ONE): return
            k = self._k(n)
            if k in seen: return
            seen.add(k); walk(n[2]); walk(n[3])
        walk(node)
        return len(seen)

    def onset(self, node, order=None):
        from itertools import product
        out = []
        for bits in product([False, True], repeat=self.nvars):
            a = {i: bits[i] for i in range(self.nvars)}
            if self.evaluate(node, a):
                out.append(''.join('1' if bits[i] else '0' for i in range(self.nvars)))
        return sorted(out)


# --- drawing what you just built ----------------------------------------
def draw(mgr, node, names=None, label=None):
    # Same convention as jove.Bdd, so the two can be compared by eye: blue
    # is the 1-branch, red the 0-branch, boxes are terminals.  Hash consing
    # is the thing you SEE here -- a shared sub-diagram is ONE node with
    # two arrows into it, not two copies of the same picture.
    import graphviz
    names = names or ['x%d' % i for i in range(mgr.nvars)]
    lines, ids, seen = [], {}, set()

    def nid(n):
        k = mgr._k(n)
        if k not in ids:
            ids[k] = 'N%d' % len(ids)
        return ids[k]

    def walk(n):
        k = mgr._k(n)
        if k in seen:
            return
        seen.add(k)
        if n in (ZERO, ONE):
            lines.append('%s [label=%d, shape=box, peripheries=2, color=%s]'
                         % (nid(n), n, 'Blue' if n else 'Red'))
            return
        lines.append('%s [label="%s", shape=circle]' % (nid(n), names[n[1]]))
        for bit, kid in ((0, n[2]), (1, n[3])):
            walk(kid)
            lines.append('%s->%s [label="%d", color=%s]'
                         % (nid(n), nid(kid), bit, 'blue' if bit else 'red'))

    walk(node)
    head = 'digraph G {\n  fontsize=12;\n  node [fontname="Helvetica"];\n'
    if label:
        head += '  label="%s"; labelloc=t; fontsize=14;\n' % label
    return graphviz.Source(head + '\n'.join('  ' + l for l in lines) + '\n}')

# NOTE on counting.  mgr.size(node) counts INTERNAL nodes; jove.Bdd's
# .nodes counts everything reachable, terminals included.  Expect the two
# to differ by up to 2, and say which you mean.

### Deciding things, once you have the BDD

In [ ]:
def is_tautology(b, g): return g is ONE
def is_unsat(b, g):     return g is ZERO
def equivalent(b, f, g): return f is g          # pointer equality!

<!-- nav-strip -->

---

&larr;&nbsp;[Ch17&nbsp;5.&nbsp;From Decision Tree to BDD: What the Construction Actually Does](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17/Concept-From-Decision-Tree-To-BDD/Concept-From-Decision-Tree-To-BDD.ipynb) &nbsp;&middot;&nbsp; [**Chapter 17** index](https://github.com/ganeshutah/Jove/blob/master/Chapter17/README.md) &nbsp;&middot;&nbsp; [Ch17&nbsp;7.&nbsp;BDD Sizes, Dynamic Reordering, and the NP-Completeness of Ordering](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17/Concept-BDD-Sizes-And-Reordering/Concept-BDD-Sizes-And-Reordering.ipynb)&nbsp;&rarr;

---

## 3. Tests

**Hash consing, visible.** Draw what `mk` and `apply` built. A shared sub-diagram is ONE node with two arrows into it &mdash; that sharing is what makes equality a pointer comparison.

In [ ]:
m = BDD(3)
a_, b_, c_ = m.var(0), m.var(1), m.var(2)
shared = m.OR(m.AND(a_, b_), c_)
draw(m, shared, ['a', 'b', 'c'],
     'built by apply(): (a & b) | c  --  %d internal nodes' % m.size(shared))

**Canonicity:** two different expressions, one node.

In [ ]:
b = BDD(3)
x, y, z = b.var(0), b.var(1), b.var(2)
f1 = b.AND(x, b.OR(y, z))
f2 = b.OR(b.AND(x, y), b.AND(x, z))            # distributive law
print("f1 is f2 ?", f1 is f2)
assert f1 is f2
print("\nNo comparison was performed.  They are the SAME OBJECT.")

Equivalence checking is therefore $O(1)$.

In [ ]:
import time
N = 14
b = BDD(N)
xs = [b.var(i) for i in range(N)]
a1 = xs[0]
for v in xs[1:]: a1 = b.AND(a1, v)
a2 = xs[-1]
for v in reversed(xs[:-1]): a2 = b.AND(v, a2)
t0 = time.time(); same = equivalent(b, a1, a2); t1 = time.time()
print("AND built left-to-right vs right-to-left, %d vars" % N)
print("   equivalent? %s  in %.7f seconds" % (same, t1 - t0))
assert same

**Tautology and unsatisfiability** are terminal-node tests.

In [ ]:
taut = b.OR(xs[0], b.NOT(xs[0]))
unsat = b.AND(xs[0], b.NOT(xs[0]))
print("x OR NOT x is a tautology ? ", is_tautology(b, taut))
print("x AND NOT x is unsat      ? ", is_unsat(b, unsat))
assert is_tautology(b, taut) and is_unsat(b, unsat)
print("\nCompare Chapter 16: SAT is NP-complete.  The hard work moved into")
print("BUILDING the BDD -- which can blow up.  Querying it is free.")

**Apply** with memoisation, and the cache doing its job.

In [ ]:
b2 = BDD(8)
ys = [b2.var(i) for i in range(8)]
before = len(b2.apply_cache)
g = ys[0]
for v in ys[1:]: g = b2.XOR(g, v)
print("parity over 8 vars : %d nodes, %d apply-cache entries"
      % (b2.size(g), len(b2.apply_cache) - before))
print("without the cache, apply would re-derive shared subgraphs repeatedly")

De Morgan and double negation, verified by pointer equality.

In [ ]:
b3 = BDD(3)
p, q = b3.var(0), b3.var(1)
assert b3.NOT(b3.NOT(p)) is p
assert b3.NOT(b3.AND(p, q)) is b3.OR(b3.NOT(p), b3.NOT(q))
assert b3.NOT(b3.OR(p, q)) is b3.AND(b3.NOT(p), b3.NOT(q))
print("NOT NOT p          is p            : verified by identity")
print("NOT (p AND q)      is NOT p OR NOT q")
print("NOT (p OR q)       is NOT p AND NOT q")
print("\nThree laws of Boolean algebra, checked with `is`.")

The chain of ideas, in one line.

In [ ]:
print("Myhill-Nerode  ->  canonical minimal DFA")
print("               ->  canonical reduced BDD (fixed variable order)")
print("hash consing   ->  canonical means IDENTICAL, not merely isomorphic")
print("               ->  equivalence checking is a pointer comparison")

## 4. Exercises


1. Why does canonicity require a **fixed** variable order?
2. What is the worst-case cost of `apply` without the cache?
3. How would you check implication $f \Rightarrow g$ with BDDs?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter17/Concept-Canonicity-And-Apply')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')